<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson02/01_python_and_training_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Python basics & the training loop

This is the first notebook in a short series:

1. **Python basics & the training loop** ← *you are here*
2. Dense neural network on FashionMNIST
3. Convolutional neural network (CNN) on FashionMNIST
4. The same with fastai
5. Fine-tuning a pretrained model

**What you will learn here:**
- just enough Python to read the code in the next notebooks
- what a *tensor* is
- what a *gradient* is (no heavy math, promise)
- how *gradient descent* finds the best parameters
- the **training loop**: the same 5 steps used to train *every* neural network, from a straight line to ChatGPT

No GPU is needed for this notebook.

### How to use a notebook

- A notebook is made of **cells**. Grey cells contain code, white cells contain text.
- Run a cell with **Shift + Enter**. The output appears below it.
- Run the cells **from top to bottom**. Later cells use variables from earlier ones.
- If things get weird: *Runtime → Restart session* and run everything again.
- You cannot break anything, so **change the code and experiment!** Cells marked **Try it** invite you to do so.

---
## 1. Python in 10 minutes

If you know Python, skip to section 2. If you know R, most of it will feel familiar. Watch out for the differences highlighted below.

### Variables and printing

In [ ]:
name = "Ada"
age = 36
height = 1.65
likes_math = True

print(name, age, height, likes_math)
print(type(name), type(age), type(height), type(likes_math))

An **f-string** (note the `f` before the quotes) lets you put variables into text:

In [ ]:
print(f"{name} is {age} years old and {height} m tall.")
print(f"Pi with 2 decimals: {3.14159:.2f}")

### Lists

A list holds several values. **Python counts from 0** (R counts from 1!).

In [ ]:
fruits = ["apple", "banana", "cherry", "date"]

print(fruits[0])     # first element
print(fruits[-1])    # last element
print(fruits[1:3])   # elements 1 and 2 (the end index is NOT included)
print(len(fruits))   # number of elements

### Loops and indentation

Python uses **indentation** (4 spaces) instead of `{ }` to mark which lines belong to a loop or function.

In [ ]:
for fruit in fruits:
    print(fruit.upper())

print("---")

for i in range(3):          # range(3) gives 0, 1, 2
    print("step", i)

### Functions

`**` means power (not `^` as in R or Excel).

In [ ]:
def square(x):
    return x ** 2

square(4)

**Try it:** Write a function `cube(x)` that returns `x` to the third power and call it on 3.

In [ ]:
# your code here

### Classes (this one matters!)

A **class** is a recipe for creating objects that hold data *and* functions. You will see classes in the next notebook, because **every neural network in PyTorch is a class**. Here is the pattern:

In [ ]:
class Dog:
    def __init__(self, name):     # runs once, when a new dog is created
        self.name = name          # store data inside the object ("self" = this dog)

    def speak(self):              # a function that belongs to the dog
        return f"{self.name} says woof!"

rex = Dog("Rex")
print(rex.name)
print(rex.speak())

Remember this shape. A neural network will look like this:

```python
class MyNetwork(nn.Module):
    def __init__(self):        # here we list the LAYERS
        ...
    def forward(self, x):      # here we say how data FLOWS through the layers
        ...
```

---
## 2. From lists to tensors

Lists are slow, and you cannot do math on all their elements at once. For numbers we use arrays: **NumPy arrays**, or in deep learning **PyTorch tensors**. Operations apply to *every element at once*.

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4])
print(a * 10)
print(a ** 2)
print(a.sum(), a.mean())

In [ ]:
import torch

t = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(t * 10)
print(t.mean())

A **tensor** is simply an n-dimensional array of numbers:

| dimensions | example | shape |
|---|---|---|
| 0 | one number | `[]` |
| 1 | a list of numbers | `[4]` |
| 2 | a grayscale image | `[28, 28]` |
| 4 | a batch of 64 grayscale images | `[64, 1, 28, 28]` |

Every tensor has a **shape**, a **dtype** (type of numbers) and a **device** (CPU or GPU). When something breaks in deep learning, it is usually the **shape**, so get used to printing it!

In [ ]:
image = torch.rand(28, 28)           # a fake 28x28 grayscale "image" with random pixels
print("shape: ", image.shape)
print("dtype: ", image.dtype)
print("device:", image.device)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image, cmap="gray")
plt.title("A random 28x28 image")
plt.show()

In [ ]:
batch = torch.rand(64, 1, 28, 28)    # 64 images, 1 color channel, 28x28 pixels
print(batch.shape)
print(batch[0].shape)                # the first image in the batch

Neural networks are much faster on a **GPU**. Tensors must be moved there explicitly. On Colab you get a GPU via *Runtime → Change runtime type → T4 GPU* (we will need it from notebook 2 on).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

---
## 3. What is a gradient?

Imagine you are standing on a hill in thick fog and want to get to the bottom. You cannot see the valley, but you **can feel the slope under your feet**. So you take a small step downhill, feel again, take another step...

The **gradient** is exactly that slope. It answers: *"If I nudge `x` a tiny bit, how much does `f(x)` change?"*

Let's take a simple function with its minimum at `x = 3`:

In [ ]:
def f(x):
    return (x - 3) ** 2

xs = np.linspace(-2, 8, 100)    # 100 numbers between -2 and 8
plt.plot(xs, f(xs))
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("f(x) = (x - 3)²,  minimum at x = 3")
plt.show()

We can measure the slope at `x = 0` by nudging `x` a tiny bit and seeing how much `f` changes:

In [ ]:
x = 0.0
nudge = 0.0001
slope = (f(x + nudge) - f(x)) / nudge
print(f"slope at x = {x}: {slope:.3f}")

The slope is about **-6**. *Negative* means that going right (increasing `x`) goes **down**. So to get lower, we should increase `x`.

**Try it:** Change `x` to `5` or `3` in the cell above. What is the sign of the slope? What happens at the minimum?

### PyTorch computes gradients for you

Nudging works for one number. A neural network has *millions* of parameters, so PyTorch computes all gradients automatically (**autograd**):

In [ ]:
x = torch.tensor(0.0, requires_grad=True)   # "PyTorch, please track gradients for x"
y = f(x)                                    # compute the function as usual
y.backward()                                # compute the gradient of y with respect to x
print("gradient:", x.grad)

Same answer: **-6**. (For the curious: the derivative of `(x-3)²` is `2·(x-3)`, and at `x = 0` that gives `-6`.)

---
## 4. Gradient descent: walking downhill

The recipe is simple. Repeat:

$$x_{new} = x - \text{learning rate} \times \text{gradient}$$

- the gradient says **which direction** is uphill, so we go the opposite way (hence the minus sign)
- the **learning rate** says **how big a step** we take

In [ ]:
x = torch.tensor(0.0, requires_grad=True)   # starting point
learning_rate = 0.1
history = [x.item()]

for step in range(20):
    y = f(x)                              # compute the function
    y.backward()                          # compute the gradient
    with torch.no_grad():                 # (the update itself should not be tracked)
        x -= learning_rate * x.grad       # take a step downhill
    x.grad.zero_()                        # reset the gradient (explained below)
    history.append(x.item())
    print(f"step {step:2d}:  x = {x.item():.4f},  f(x) = {f(x).item():.4f}")

In [ ]:
plt.plot(xs, f(xs))
plt.plot(history, [f(h) for h in history], "o-", color="red")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Gradient descent path")
plt.show()

**Try it:** Rerun the two cells above with a different `learning_rate`:
- `0.01`: too small, very slow
- `0.9`: jumps back and forth over the minimum
- `1.1`: **explodes!**

The learning rate is the single most important knob in training neural networks.

### Why `x.grad.zero_()`?

PyTorch **adds** each new gradient to the old one instead of replacing it. If we forget to reset, the gradients pile up:

In [ ]:
x = torch.tensor(0.0, requires_grad=True)
f(x).backward()
print("after 1st backward:", x.grad)
f(x).backward()
print("after 2nd backward:", x.grad, " <- wrong! it added up")

---
## 5. Training = gradient descent on the loss

Now the key insight. **Training a model is exactly the same thing**:
- instead of one number `x`, we have the model's **parameters** (weights)
- instead of `f(x)`, we minimize the **loss**, which measures how wrong the model's predictions are

Let's fit a line to data, like the demo from the last lecture. Our data: hours studied vs. exam points for 100 students.

In [ ]:
torch.manual_seed(0)                                # so that we all get the same random numbers

hours = torch.rand(100) * 10                        # 100 students studied 0-10 hours
points = 5 * hours + 20 + torch.randn(100) * 5      # the "true" rule: 20 + 5 points per hour, plus noise

plt.scatter(hours, points)
plt.xlabel("hours studied")
plt.ylabel("exam points")
plt.show()

Our **model** is a line: `prediction = m * hours + b`. It has two **parameters**, `m` and `b`. We start with a bad guess (both zero) and let gradient descent find better values.

Our **loss** is the *mean squared error*: the average of (prediction − truth)².

In [ ]:
m = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

def model(x):
    return m * x + b

def mse_loss(prediction, target):
    return ((prediction - target) ** 2).mean()

loss = mse_loss(model(hours), points)
print("loss with m = 0, b = 0:", loss.item())

Let's peek at the gradients. Both are negative, which means *"increase me and the loss goes down"*:

In [ ]:
loss.backward()
print("gradient for m:", m.grad.item())
print("gradient for b:", b.grad.item())
m.grad.zero_()
b.grad.zero_();

### The training loop

This is **the most important cell of today's lecture**. Every neural network is trained with these same five steps, repeated many times:

In [ ]:
learning_rate = 0.02

for epoch in range(1000):
    prediction = model(hours)               # 1. FORWARD:  make predictions
    loss = mse_loss(prediction, points)     # 2. LOSS:     how wrong are we?
    loss.backward()                         # 3. BACKWARD: compute gradients
    with torch.no_grad():                   # 4. UPDATE:   step downhill
        m -= learning_rate * m.grad
        b -= learning_rate * b.grad
    m.grad.zero_()                          # 5. RESET:    clear the gradients
    b.grad.zero_()

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}:  loss = {loss.item():8.2f},  m = {m.item():.2f},  b = {b.item():.2f}")

print(f"\nlearned: m = {m.item():.2f}, b = {b.item():.2f}   (true rule: m = 5, b = 20)")

In [ ]:
plt.scatter(hours, points, label="data")
plt.plot(hours, model(hours).detach(), color="red", label="learned line")
plt.xlabel("hours studied")
plt.ylabel("exam points")
plt.legend()
plt.show()

We found (almost) the true rule, just from data! It is not exactly 5 and 20 because of the random noise: this is the best line for *these particular* 100 students.

A few words you will hear all the time:
- **parameters / weights**: the numbers the model learns (here `m` and `b`)
- **epoch**: one pass over all training data
- **learning rate**: the step size

**Try it:** Rerun the loop (start from the `m = ...` cell) with `learning_rate = 0.001` and then `0.05`. What happens? (`nan` means "not a number": the values grew so large that the computer gave up.)

---
## 6. The same thing, the PyTorch way

Updating each parameter by hand is fine for 2 parameters but not for 2 million. PyTorch has ready-made building blocks:

| by hand | PyTorch |
|---|---|
| `m * x + b` | `nn.Linear(1, 1)` |
| `mse_loss(...)` | `nn.MSELoss()` |
| `m -= lr * m.grad` for every parameter | `optimizer.step()` |
| `m.grad.zero_()` for every parameter | `optimizer.zero_grad()` |

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)    # prediction = weight * x + bias
print(model)

### Look inside the model

A model is not a black box: its parameters are just tensors. Our "line" has a **weight** (that's `m`) and a **bias** (that's `b`). PyTorch initializes them randomly:

In [ ]:
print("weight:", model.weight)
print("bias:  ", model.bias)

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:8s} shape = {str(list(param.shape)):8s} count = {param.numel()}")

total = sum(p.numel() for p in model.parameters())
print("total number of parameters:", total)

Remember this cell! In the next notebook we will use it to count the parameters of a real neural network (spoiler: around 100,000).

One detail: `nn.Linear` expects data as a table with shape `(number of samples, number of features)`. We have 100 samples with 1 feature each, so we reshape `[100]` into `[100, 1]`:

In [ ]:
X = hours.unsqueeze(1)     # shape [100] -> [100, 1]
Y = points.unsqueeze(1)
print(hours.shape, "->", X.shape)

And now the training loop. **Compare it with the one in section 5**: same steps, less typing.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)   # SGD = (stochastic) gradient descent

for epoch in range(1000):
    prediction = model(X)            # 1. FORWARD
    loss = loss_fn(prediction, Y)    # 2. LOSS
    optimizer.zero_grad()            # 5. RESET (by convention done before backward)
    loss.backward()                  # 3. BACKWARD
    optimizer.step()                 # 4. UPDATE

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}:  loss = {loss.item():8.2f}")

print(f"\nlearned: weight = {model.weight.item():.2f}, bias = {model.bias.item():.2f}   (true rule: 5 and 20)")

---
## 7. Summary: the recipe

Every time we train a neural network, we need five ingredients:

| ingredient | today | next notebook |
|---|---|---|
| **data** | 100 (hours, points) pairs | 60,000 images of clothes |
| **model** | a line, 2 parameters | a neural network, ~100,000 parameters |
| **loss** | mean squared error | cross-entropy (for classification) |
| **optimizer** | SGD | SGD |
| **training loop** | forward → loss → backward → update → reset | **exactly the same!** |

If you understand the training loop in section 6, you can read the code of the next notebooks.

### Exercises
1. In section 6, train for only 100 epochs. Is the line good enough? What about 5000 epochs?
2. Change the "true rule" in section 5 to `points = 3 * hours + 50 + ...` and retrain. Does the model find it?
3. **Bonus:** Replace `torch.optim.SGD` with `torch.optim.Adam(model.parameters(), lr=0.5)`. Adam is a smarter optimizer. How many epochs does it need?